# 🚁 Drone CV — YOLOv8 Training at imgsz=1280 (A100)

**Author:** Richard Wirén, Lead Solution Architect 
**Repo:** [github.com/rwiren/drone-cv-detection](https://github.com/rwiren/drone-cv-detection) 
**Runtime:** Colab Pro — A100 GPU

---

### Setup: Google Drive Structure
Before running, create this folder in your Drive:
```
My Drive/DroneCV/
├── autel_labels.zip    (21KB, from repo notebooks/)
└── results/            (auto-created, stores trained models)
```

VisDrone dataset auto-downloads (~2GB). Trained model saved back to Drive.

In [ ]:
#@title 1. Mount Drive & Install
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics

import torch, os
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.0f} GB')

# Create results dir on Drive
DRIVE_DIR = '/content/drive/MyDrive/DroneCV'
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')
print(f'Contents: {os.listdir(DRIVE_DIR)}')

In [ ]:
#@title 2. Prepare Autel Campus Labels
import zipfile

autel_zip = f'{DRIVE_DIR}/autel_labels.zip'
if os.path.exists(autel_zip):
    with zipfile.ZipFile(autel_zip, 'r') as z:
        z.extractall('/content/autel_data/')
    n_labels = len([f for f in os.listdir('/content/autel_data/labels/train') if f.endswith('.txt')])
    print(f'✅ Autel labels extracted: {n_labels} files')
else:
    print(f'⚠️ Upload autel_labels.zip to {DRIVE_DIR}/')
    print('  (Get it from: github.com/rwiren/drone-cv-detection/notebooks/autel_labels.zip)')

In [ ]:
#@title 3. Train YOLOv8s — imgsz=1280, 30 epochs
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

# VisDrone auto-downloads on first use (~2GB)
results = model.train(
    data='VisDrone.yaml',
    epochs=30,
    imgsz=1280,
    batch=16,           # A100 80GB: batch=16 at 1280px
    device=0,
    workers=4,
    patience=10,
    project='/content/runs',
    name='visdrone_1280',
    exist_ok=True,
    mosaic=1.0,
    mixup=0.1,
    cos_lr=True,
    plots=True,
    save=True,
)

print(f'\n✅ Training complete: {results.save_dir}')

In [ ]:
#@title 4. Evaluate & Report
model = YOLO(f'{results.save_dir}/weights/best.pt')
metrics = model.val(data='VisDrone.yaml', imgsz=1280)

print('═' * 50)
print('  TRAINING RESULTS — YOLOv8s @ imgsz=1280')
print('═' * 50)
print(f'  mAP50 (all classes):  {metrics.box.map50:.3f}')
print(f'  mAP50-95 (all):       {metrics.box.map:.3f}')
print(f'  mAP50 car:            {metrics.box.maps[3]:.3f}')
print(f'  mAP50 pedestrian:     {metrics.box.maps[0]:.3f}')
print(f'  mAP50 van:            {metrics.box.maps[4]:.3f}')
print(f'  mAP50 truck:          {metrics.box.maps[5]:.3f}')
print('═' * 50)

In [ ]:
#@title 5. Save to Drive
import shutil

# Copy best weights and training plots to Drive
best_src = f'{results.save_dir}/weights/best.pt'
best_dst = f'{DRIVE_DIR}/results/visdrone_yolov8s_1280_best.pt'
shutil.copy(best_src, best_dst)

# Copy training curves
for f in ['results.csv', 'confusion_matrix.png', 'results.png']:
    src = f'{results.save_dir}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_DIR}/results/{f}')

print(f'✅ Model saved: {best_dst}')
print(f'   Size: {os.path.getsize(best_dst)/1e6:.1f} MB')
print(f'\nDownload from Drive or use directly in next session.')

In [ ]:
#@title 6. (Optional) Quick Test on Sample Image
# Upload a test image or use a VisDrone val sample
import glob

val_imgs = glob.glob('/content/datasets/VisDrone/images/val/*.jpg')[:3]
if val_imgs:
    model = YOLO(best_dst)
    for img in val_imgs:
        r = model(img, conf=0.25, imgsz=1280, verbose=False)[0]
        classes = {}
        for b in r.boxes:
            c = r.names[int(b.cls)]
            classes[c] = classes.get(c, 0) + 1
        print(f'{os.path.basename(img)}: {sum(classes.values())} detections {dict(classes)}')